# Pipelines

`pipeline(task)` returns a function. Call it with the input. This is inference on a model that is already trained.

`HF_TOKEN` comes from the repo `.env`. These cells match `transformers==4.57.6`, `diffusers==0.32.2`, and `datasets==3.6.0`. A CUDA GPU is required. The image and the wav file are written in this folder.


## Setup

In [ ]:
import os
from pathlib import Path

import torch
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
day_dir = repo_root / "W3-Open Source Models" / "Day-2"
load_dotenv(repo_root / ".env", override=True)

hf_home = repo_root / "W3-Open Source Models" / "hf-cache"
hf_home.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_home)
os.environ["HF_HUB_CACHE"] = str(hf_home / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_home / "datasets")

from huggingface_hub import login
from IPython.display import display
from transformers import pipeline

login(os.getenv("HF_TOKEN"))
print("Model cache:", hf_home)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU")

device = "cuda"
dtype = torch.float16
print(torch.cuda.get_device_name(0))


## Sentiment

The default model, then `nlptown/bert-base-multilingual-uncased-sentiment`, which scores from 1 to 5 stars.

In [ ]:
sentiment = pipeline("sentiment-analysis", device=device)
print(sentiment("I'm super excited to be on the way to LLM mastery!"))
print(sentiment("I should be more excited to be on the way to LLM mastery!"))

better_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device,
)
print(better_sentiment("I should be more excited to be on the way to LLM mastery!!"))

del sentiment, better_sentiment
torch.cuda.empty_cache()


## Named entities

In [ ]:
ner = pipeline("ner", device=device)
text = (
    "AI Engineers are learning about the amazing pipelines from HuggingFace "
    "in Google Colab from Ed Donner"
)
for entity in ner(text):
    print(entity)

del ner
torch.cuda.empty_cache()


## Question answering

The answer has to come from the context string.

In [ ]:
question = "What are Hugging Face pipelines?"
context = "Pipelines are a high level API for inference of LLMs with common tasks"
answerer = pipeline("question-answering", device=device)
print(answerer(question=question, context=context))

del answerer
torch.cuda.empty_cache()


## Summarization

In [ ]:
summarizer = pipeline("summarization", device=device)
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""
summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]["summary_text"])

del summarizer
torch.cuda.empty_cache()


## Translation

English to French with the default model, then English to Spanish with `Helsinki-NLP/opus-mt-en-es`. Other translation models are listed under the [translation task](https://huggingface.co/models?pipeline_tag=translation&sort=trending).

In [ ]:
sentence = (
    "The Data Scientists were truly amazed by the power and simplicity "
    "of the HuggingFace pipeline API."
)

translator = pipeline("translation_en_to_fr", device=device)
print(translator(sentence)[0]["translation_text"])
del translator

translator = pipeline(
    "translation_en_to_es",
    model="Helsinki-NLP/opus-mt-en-es",
    device=device,
)
print(translator(sentence)[0]["translation_text"])

del translator
torch.cuda.empty_cache()


## Zero-shot classification

In [ ]:
classifier = pipeline("zero-shot-classification", device=device)
result = classifier(
    "Hugging Face's Transformers library is amazing!",
    candidate_labels=["technology", "sports", "politics"],
)
print(result)

del classifier
torch.cuda.empty_cache()


## Text generation

In [ ]:
generator = pipeline("text-generation", device=device)
result = generator(
    "If there's one thing I want you to remember about using HuggingFace pipelines, it's"
)
print(result[0]["generated_text"])

del generator
torch.cuda.empty_cache()


## Image

SDXL Turbo, four steps. Same call as Day 1, through the diffusers pipeline.

In [ ]:
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=dtype,
    variant="fp16",
)
pipe.to(device)

prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
image.save(day_dir / "sdxl-turbo.png")
display(image)

del pipe
torch.cuda.empty_cache()


## Speech

SpeechT5, with one speaker embedding from the CMU Arctic set.

In [ ]:
import soundfile as sf
from datasets import load_dataset
from IPython.display import Audio

synthesizer = pipeline("text-to-speech", "microsoft/speecht5_tts", device=device)
embeddings = load_dataset(
    "matthijs/cmu-arctic-xvectors",
    split="validation",
    trust_remote_code=True,
)
speaker_embedding = torch.tensor(embeddings[7306]["xvector"]).unsqueeze(0)
speech = synthesizer(
    "Hi to an artificial intelligence engineer, on the way to mastery!",
    forward_params={"speaker_embeddings": speaker_embedding},
)

path = day_dir / "speech.wav"
sf.write(path, speech["audio"], speech["sampling_rate"])
display(Audio(path))

del synthesizer
torch.cuda.empty_cache()


## Other tasks

Transformer tasks: [pipelines](https://huggingface.co/docs/transformers/main_classes/pipelines).

Diffusion tasks: [pipeline overview](https://huggingface.co/docs/diffusers/en/api/pipelines/overview).
